# VoiceOps Studio — batch Colab launcher

This notebook is a thin launcher. The canonical workflow source of truth is `elevenlabs_api.py` in this repository.

Runtime files are pinned to a reviewed commit and checked by SHA-256 before execution. To update, review the new commit and update both GITHUB_REF and the expected hashes in this launcher.


In [ ]:
import hashlib
import pathlib
import runpy
import tempfile
import urllib.request

GITHUB_OWNER = "Just9120"
GITHUB_REPO = "voiceops-studio"
GITHUB_REF = "7382c2f788393d76264c4edcc6e77b6656a18bb3"  # reviewed immutable release

if len(GITHUB_REF) != 40 or any(c not in "0123456789abcdef" for c in GITHUB_REF):
    raise RuntimeError("A reviewed full commit SHA is required for Colab runtime.")

RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_REF}"
ARTIFACT_SHA256 = {
    "elevenlabs_api.py": "15510b89ff2ab2f6e824d44431b971fdd5d7a84e49b61ca684054c42d56dbbfd",
    "colab_stt_transport.py": "3054b90597364db78dea7fdcea559b8fd288c9c00904bb68738c40ed8bbd8a06",
    "scripts/check_colab_dependency_install.py": "f6dc92dd4448bbcdca8e3668dce9c29cc8be9c527e00bf9f8bde1f8b1c9dc28a",
    "requirements-colab-linux-x64-py3.12.lock": "ebd0e097b0064eb4f4885fc2f6fa2ed993c596004bbdd08dac7cc70886ee57a0",
    "requirements-colab-linux-x64-py3.13.lock": "64d64577414306fb613cd2ce933b408d827107beef52aac1d97f99cf5187199e",
}


def download_verified(url, destination, expected_sha256):
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = response.read(2 * 1024 * 1024 + 1)
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise RuntimeError("Downloaded Colab runtime does not match the reviewed release; nothing will execute.")
    destination.write_bytes(payload)


def launch():
    work = tempfile.TemporaryDirectory(prefix="voiceops-colab-batch-")
    directory = pathlib.Path(work.name)

    def artifact(name):
        destination = directory / pathlib.Path(name).name
        download_verified(f"{RAW_BASE}/{name}", destination, ARTIFACT_SHA256[name])
        return destination

    try:
        src_path = artifact("elevenlabs_api.py")
        installer = runpy.run_path(str(artifact("scripts/check_colab_dependency_install.py")))
        lock = artifact(installer["colab_lock_name"]())
        interpreter = installer["install_locked_colab_python"](lock, directory / "python-env")
        transport = runpy.run_path(str(artifact("colab_stt_transport.py")))

        def stt_post(url, **kwargs):
            if not pathlib.Path(work.name).is_dir():
                raise RuntimeError("Изолированное окружение Batch недоступно. Перезапустите ячейку.")
            return transport["post_with_interpreter"](interpreter, url, **kwargs)

        source = src_path.read_text(encoding='utf-8')
        lines = source.splitlines()
        sanitized_lines = []
        removed_known_pip_line = False
        expected_pip_markers = (
            'google-api-python-client',
            'google-auth-httplib2',
            'google-auth-oauthlib',
            'requests',
            'ipywidgets',
        )
        for idx, line in enumerate(lines, start=1):
            stripped = line.lstrip()
            if stripped.startswith('!') or stripped.startswith('%'):
                is_expected_top_level_pip = (
                    stripped.startswith('!pip ')
                    and idx <= 20
                    and any(marker in stripped for marker in expected_pip_markers)
                )
                if is_expected_top_level_pip and not removed_known_pip_line:
                    removed_known_pip_line = True
                    print(f"Removed known Colab dependency install line at {idx}: {stripped}")
                    continue
                if stripped.startswith('!pip '):
                    raise RuntimeError(
                        f"Unexpected !pip line in elevenlabs_api.py at line {idx}: {line!r}. \
        Only the known top-level Colab dependency install line is allowed."
                    )
                raise RuntimeError(
                    f"Unsupported notebook magic/shell line in elevenlabs_api.py at line {idx}: {line!r}. \
        Refuse to guess. Please sanitize upstream file or run directly in Colab."
                )
            sanitized_lines.append(line)

        exec_globals = {'__name__': '__main__', '__file__': str(src_path), 'VOICEOPS_STT_POST': stt_post}
        exec(compile('\n'.join(sanitized_lines) + '\n', str(src_path), 'exec'), exec_globals)
    except BaseException:
        work.cleanup()
        raise


print(f"Using GitHub ref: {GITHUB_REF}")
launch()
